In [1]:
import yfinance as yf
import pandas as pd

df = yf.download("AAPL", start = "2018-01-01")#create dataframe containing stock data from date entered to current date
print(df.head())# print first 5 values of dataframe
print (df.shape)# pulls the dimensions of the dataframe which in this case tells us how many trading days there have been and how many features there are

[*********************100%***********************]  1 of 1 completed

Price           Close       High        Low       Open     Volume
Ticker           AAPL       AAPL       AAPL       AAPL       AAPL
Date                                                             
2018-01-02  40.304169  40.313530  39.602250  39.812828  102223600
2018-01-03  40.297150  40.839968  40.233980  40.367342  118071600
2018-01-04  40.484333  40.587282  40.262059  40.369685   89738400
2018-01-05  40.945259  41.031828  40.489013  40.580262   94640000
2018-01-08  40.793190  41.087995  40.694918  40.793190   82271200
(2074, 5)


In [2]:
df["Return"] = df["Close"].pct_change()# as returns are more useful than prices, this formula calculates the percentage change in closing price from one day to the next and stores it
print(df["Return"].head(10))# prints the first 10 values of the return column alongside the percentage change
df["Target"] = (df["Return"].shift(-1)>0).astype(int)# creates a target variable indicating whether the stock will go up (1) or down (0) the next day
print(df["Target"].value_counts())# prints the count of each target value

Date
2018-01-02         NaN
2018-01-03   -0.000174
2018-01-04    0.004645
2018-01-05    0.011385
2018-01-08   -0.003714
2018-01-09   -0.000115
2018-01-10   -0.000230
2018-01-11    0.005680
2018-01-12    0.010326
2018-01-16   -0.005082
Name: Return, dtype: float64
Target
1    1108
0     966
Name: count, dtype: int64


In [3]:
df["MA10"] = df["Close"].rolling(10).mean()# averages the closing price for the last 10 days
df["MA50"] = df["Close"].rolling(50).mean()# averages the closing price for the last 50 days
# produces two values the can be comapred to determine if stock trend is bullish(+) or bearish(-)
print(df[["Close", "MA10", "MA50"]].tail(10))# prints the last 10 values of the closing price and the two moving averages to see how they compare


Price            Close        MA10        MA50
Ticker            AAPL                        
Date                                          
2026-03-20  247.990005  254.134000  261.027877
2026-03-23  251.490005  253.295000  260.881720
2026-03-24  251.639999  252.376001  260.731969
2026-03-25  252.619995  251.557001  260.584235
2026-03-26  252.889999  251.270001  260.425916
2026-03-27  248.800003  251.138002  260.207576
2026-03-30  246.630005  250.519002  259.980804
2026-03-31  253.789993  250.475002  259.950782
2026-04-01  255.630005  251.044002  260.133994
2026-04-02  255.919998  251.740001  260.304025


In [4]:
df["Volatility"] = df["Return"].rolling(10).std()#the standard deviation function measures how spread out the returns have been in the last 10 days
# a low volatility would indicate that there isn't much difference between the percentage changes from day to day perspective while a high volatility would indicate the opposite
# this is important because volatility also has momentum which means that a chaotic period will tend to stay chaotic while a calm period will tend to stay calm
df["Momentum"] = df["Close"]-df["Close"].shift(10) # difference between the closing price currently and the closing price 10 days ago.
#A positive number indicates upwards momentum while a negative number indicates downwards momentum
print(df[["Return","MA10", "MA50", "Volatility", "Momentum"]].tail(10))#new print including volatility and momentum


Price         Return        MA10        MA50 Volatility  Momentum
Ticker                                                           
Date                                                             
2026-03-20 -0.003896  254.134000  261.027877   0.011976 -9.469986
2026-03-23  0.014113  253.295000  260.881720   0.012623 -8.389999
2026-03-24  0.000596  252.376001  260.731969   0.012474 -9.189987
2026-03-25  0.003894  251.557001  260.584235   0.012658 -8.190002
2026-03-26  0.001069  251.270001  260.425916   0.011322 -2.869995
2026-03-27 -0.016173  251.138002  260.207576   0.010209 -1.319992
2026-03-30 -0.008722  250.519002  259.980804   0.009665 -6.190002
2026-03-31  0.029031  250.475002  259.950782   0.013790 -0.440002
2026-04-01  0.007250  251.044002  260.133994   0.012585  5.690002
2026-04-02  0.001134  251.740001  260.304025   0.012406  6.959991


In [5]:
#RSI Strategy - The Relative Strength index assigns a value to measure the recent performance of the stock. It is calculated by comparing the average gains to the average losses
# The RSI ranges from 0 to 100, with values above 70 indicating that the stock is overbought and values below 30 indicating that the stock is oversold 
# Anything around 50 indicates that the stock is neutral
#  
delta = df["Close"].diff()# calculates the difference in closing price from one day to the next
gain = delta.where(delta>0,0)# records the postive returns and sets the negetive ones to 0
loss = -delta.where(delta<0,0)# Records the negative returns and sets the positives to 0

avg_gain = gain.rolling(14).mean()# Average gain pver the last 14 days
avg_loss = loss.rolling(14).mean()# Average loss over the last 14 days

rs = avg_gain / avg_loss # relative strength ratio of average gain to average loss
df["RSI"] = 100 - (100 / (1 + rs))# calculates the RSI value using the formula
print(df["RSI"].tail(10))# print the last 10 values in the RSI column

Date
2026-03-20    23.603657
2026-03-23    32.086500
2026-03-24    33.584792
2026-03-25    37.974260
2026-03-26    42.209348
2026-03-27    32.129003
2026-03-30    27.964006
2026-03-31    41.082305
2026-04-01    49.820208
2026-04-02    59.415598
Name: RSI, dtype: float64


In [6]:
#MACD Strategy - The Moving Average Convergence Divergence is similar to the moving average stratgey (MA10, MA50) but it uses exponential averages instead of simple ones
# All this does is just weighs in recent days more heavily to capture the recent trends
# All it is is the 12-day exponential average - 26-day exponential average
# positive when short term average is larger than long term(bullish) and negative when short term averge is smaller than long term (bearish)
exp12 = df["Close"].ewm(span = 12, adjust = False).mean()# EWM stands for expontentially weighted mean
exp26 = df["Close"].ewm(span = 26, adjust = False).mean()

df["MACD"] = exp12 - exp26# This signal is much more responsive to recent price changes 
df["MACD_Signal"] = df["MACD"].ewm(span = 9, adjust=False).mean()# this signal reflects a much smoother change as it is the average of the MACD over the past 9 days
print(df[["MACD", "MACD_Signal"]].tail(10))

Price           MACD MACD_Signal
Ticker                          
Date                            
2026-03-20 -4.183685   -2.821722
2026-03-23 -4.107600   -3.078898
2026-03-24 -3.989214   -3.260961
2026-03-25 -3.772824   -3.363334
2026-03-26 -3.538753   -3.398417
2026-03-27 -3.641304   -3.446995
2026-03-30 -3.853260   -3.528248
2026-03-31 -3.404243   -3.503447
2026-04-01 -2.866872   -3.376132
2026-04-02 -2.390052   -3.178916


In [7]:
df=df.dropna()# drops any row that has NaN in it 
print(df.shape)

(2025, 14)


In [8]:
print(df[["Return", "MA10", "MA50", "Volatility", "Momentum", "RSI", "MACD", "MACD_Signal", "Target"]].tail(5))


Price         Return        MA10        MA50 Volatility  Momentum        RSI  \
Ticker                                                                         
Date                                                                           
2026-03-27 -0.016173  251.138002  260.207576   0.010209 -1.319992  32.129003   
2026-03-30 -0.008722  250.519002  259.980804   0.009665 -6.190002  27.964006   
2026-03-31  0.029031  250.475002  259.950782   0.013790 -0.440002  41.082305   
2026-04-01  0.007250  251.044002  260.133994   0.012585  5.690002  49.820208   
2026-04-02  0.001134  251.740001  260.304025   0.012406  6.959991  59.415598   

Price           MACD MACD_Signal Target  
Ticker                                   
Date                                     
2026-03-27 -3.641304   -3.446995      0  
2026-03-30 -3.853260   -3.528248      1  
2026-03-31 -3.404243   -3.503447      1  
2026-04-01 -2.866872   -3.376132      1  
2026-04-02 -2.390052   -3.178916      0  


In [10]:
features = ["MA10", "MA50", "Volatility", "Momentum", "RSI", "MACD", "MACD_Signal"]# list of features to be used in the model

x = df[features]# creates a new dataframe with just the features
y = df["Target"]# creates a new dataframe with just the target variable

split = int(len(df)*.8)# splits the data into 80% training and 20% testing

X_train, X_test = x.iloc[:split], x.iloc[split:]# creates the training and testing sets for the features
y_train, y_test = y.iloc[:split], y.iloc[split:]# creates the training and testing sets for the target variable

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")


Training rows: 1620
Testing rows: 405


In [17]:
# The model I am using to train this data is one called XGBoost which is tree based and uses that to make decisions.
import sys
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "xgboost"])
subprocess.run([sys.executable, "-m", "pip", "install", "scikit-learn"])
print("Libraries Installed")



Libraries Installed


In [18]:
from xgboost import XGBClassifier
model = XGBClassifier(n_estimators = 100, learning_rate = .1, max_depth = 3, random_state = 42)# creates an instance of the XGBClassifier with specified parameters
model.fit(X_train, y_train)# fits the model to the training data
print("Model Trained Succesfully")

ImportError: sklearn needs to be installed in order to use this module